# Tutorial: Lorenz Attractor End-to-End Walkthrough

This notebook walks through ATT's core pipeline step by step:

1. **Generate** a chaotic time series (Lorenz system)
2. **Embed** it using Takens' theorem to reconstruct the attractor
3. **Compute** persistent homology to extract topological features
4. **Visualize** the results (persistence diagrams, barcodes, images)
5. **Detect binding** between two coupled Lorenz systems
6. **Test significance** with phase-randomized surrogates

No prior knowledge of topology or dynamical systems is assumed.

## Background

**Chaotic attractors** are geometric shapes that chaotic systems trace out in their state space. The Lorenz attractor (the famous "butterfly") lives in 3D, but in real experiments we often only observe a single scalar measurement (e.g., one EEG channel). **Takens' theorem** tells us we can reconstruct the attractor's topology from that single scalar by creating delayed copies of the signal.

**Persistent homology** is a tool from topological data analysis (TDA) that counts "holes" in data at multiple scales:
- **H0**: connected components (clusters)
- **H1**: loops (tunnels through the data)
- **H2**: voids (enclosed cavities)

These topological features are robust to noise and deformation, making them ideal summaries of attractor geometry.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from att.config import set_seed
from att.synthetic import lorenz_system, coupled_lorenz
from att.embedding import TakensEmbedder, validate_embedding
from att.topology import PersistenceAnalyzer
from att.binding import BindingDetector
from att.viz import (
    plot_persistence_diagram,
    plot_barcode,
    plot_attractor_3d,
    plot_persistence_image,
    plot_binding_image,
    plot_surrogate_distribution,
)

set_seed(42)
print("ATT loaded successfully.")

## Step 1: Generate a Lorenz trajectory

The Lorenz system is a set of three coupled ODEs that produce chaotic dynamics:

$$\dot{x} = \sigma(y - x), \quad \dot{y} = x(\rho - z) - y, \quad \dot{z} = xy - \beta z$$

With the standard parameters ($\sigma=10, \rho=28, \beta=8/3$), the system is chaotic and traces out the famous butterfly-shaped attractor.

In [ ]:
trajectory = lorenz_system(n_steps=8000, dt=0.01)
print(f"Trajectory shape: {trajectory.shape}  (timesteps x variables)")

fig, axes = plt.subplots(3, 1, figsize=(10, 5), sharex=True)
labels = ["x(t)", "y(t)", "z(t)"]
for i, (ax, label) in enumerate(zip(axes, labels)):
    ax.plot(trajectory[:, i], linewidth=0.5, color=f"C{i}")
    ax.set_ylabel(label)
axes[-1].set_xlabel("Time step")
axes[0].set_title("Lorenz system: three state variables")
plt.tight_layout()
plt.show()

In practice, we often only observe **one** of these variables. Let's pretend we only have $x(t)$:

In [ ]:
x = trajectory[:, 0]
print(f"Time series length: {len(x)}")

## Step 2: Takens embedding

**Takens' theorem** says: from a single scalar time series $x(t)$, we can reconstruct a topologically equivalent version of the attractor by forming delay vectors:

$$\mathbf{v}(t) = [x(t), \; x(t + \tau), \; x(t + 2\tau), \; \ldots, \; x(t + (d-1)\tau)]$$

Two parameters must be chosen:
- **Delay** ($\tau$): estimated via **Average Mutual Information** (AMI) --- pick the first minimum
- **Dimension** ($d$): estimated via **False Nearest Neighbors** (FNN) --- pick where FNN fraction drops to ~0

ATT's `TakensEmbedder` handles both automatically:

In [ ]:
embedder = TakensEmbedder(delay="auto", dimension="auto")
cloud = embedder.fit_transform(x)

print(f"Estimated delay (tau): {embedder.delay_}")
print(f"Estimated dimension (d): {embedder.dimension_}")
print(f"Point cloud shape: {cloud.shape}  (points x embedding dimension)")

Let's verify the embedding quality. A good embedding has a low **condition number** (ratio of largest to smallest singular values). High condition numbers mean the point cloud is nearly flat in some direction --- a sign of a degenerate embedding.

In [ ]:
quality = validate_embedding(cloud)
print(f"Condition number: {quality['condition_number']:.1f}")
print(f"Effective rank: {quality['effective_rank']:.2f}")
print(f"Degenerate: {quality['degenerate']}")

Now let's visualize the reconstructed attractor:

In [ ]:
fig = plot_attractor_3d(cloud, backend="matplotlib")
plt.title("Reconstructed Lorenz attractor (Takens embedding)")
plt.show()

## Step 3: Persistent homology

Now we compute the **persistence diagram** --- a summary of all topological features (components, loops, voids) at every spatial scale.

Each feature has a **birth** time (when it appears) and a **death** time (when it merges or fills in). Long-lived features (far from the diagonal) represent real structure; short-lived features near the diagonal are noise.

In [ ]:
pa = PersistenceAnalyzer(max_dim=2, backend="ripser")
result = pa.fit_transform(cloud, subsample=1500, seed=42)

for dim in range(3):
    n_features = len(result["diagrams"][dim])
    print(f"H{dim} features: {n_features}")
entropies = result["persistence_entropy"]
for dim, e in enumerate(entropies):
    print(f"H{dim} persistence entropy: {e:.4f}")

The Lorenz attractor should show:
- **H0**: Many short-lived components that quickly merge into one (the attractor is connected)
- **H1**: One or two dominant loops (the two butterfly wings)
- **H2**: Near zero (no enclosed voids)

Let's visualize the persistence diagram and barcode:

In [ ]:
fig = plot_persistence_diagram(result["diagrams"])
plt.show()

fig = plot_barcode(result["diagrams"])
plt.show()

### Persistence images

Persistence diagrams are sets of variable size, which makes them hard to compare. A **persistence image** converts the diagram into a fixed-size grid (a vector) by placing Gaussian kernels on each birth-persistence point. This vectorization enables subtraction, distance computation, and machine learning.

In [ ]:
images = pa.to_image(resolution=50, sigma=0.5)
print(f"Number of images: {len(images)} (one per homology dimension)")
print(f"Image shape: {images[0].shape}")

# Show the H1 persistence image
fig = plot_persistence_image([images[1]])
plt.show()

## Step 4: Binding detection

Now for ATT's core novelty: **topological binding detection**.

Given two coupled signals $x(t)$ and $y(t)$, we compute three embeddings:
1. **Marginal X**: Takens embedding of $x(t)$ alone
2. **Marginal Y**: Takens embedding of $y(t)$ alone
3. **Joint**: Joint embedding of both signals together

If the systems are coupled, the joint embedding contains topological features (loops, components) that are absent from *both* marginals. The **binding score** quantifies this emergent topology.

Let's generate two coupled Lorenz systems and detect binding:

In [ ]:
# Generate coupled Lorenz with moderate coupling
ts_x, ts_y = coupled_lorenz(n_steps=8000, coupling=2.0, seed=42)
print(f"System 1 shape: {ts_x.shape}, System 2 shape: {ts_y.shape}")

signal_x = ts_x[:, 0]  # x-component of system 1
signal_y = ts_y[:, 0]  # x-component of system 2

# Detect binding
detector = BindingDetector(max_dim=1, baseline="max")
detector.fit(signal_x, signal_y, subsample=1500, seed=42)

score = detector.binding_score()
print(f"\nBinding score: {score:.2f}")
print("  (positive = emergent topology in joint embedding)")

In [ ]:
# Visualize the binding image (residual persistence image)
fig = plot_binding_image(detector.binding_image())
plt.show()

## Step 5: Significance testing

A positive binding score could arise from finite-sample effects. We test significance using **phase-randomized surrogates**: the second signal's Fourier phases are shuffled (destroying nonlinear coupling) while preserving its power spectrum. If the observed binding score exceeds the surrogate distribution, the coupling is statistically significant.

In [ ]:
sig_result = detector.test_significance(
    n_surrogates=19,
    method="phase_randomize",
    seed=42,
    subsample=1500,
)

print(f"Observed score:  {sig_result['observed_score']:.2f}")
print(f"Surrogate mean:  {np.mean(sig_result['surrogate_scores']):.2f}")
print(f"p-value:         {sig_result['p_value']:.4f}")
print(f"Significant:     {sig_result['significant']}")

In [ ]:
fig = plot_surrogate_distribution(
    sig_result["observed_score"],
    sig_result["surrogate_scores"],
)
plt.show()

## Step 6: Uncoupled baseline

For comparison, let's check what happens with **uncoupled** systems (coupling = 0). The binding score should be lower and the significance test should fail to reject the null:

In [ ]:
ts_ux, ts_uy = coupled_lorenz(n_steps=8000, coupling=0.0, seed=42)
ux = ts_ux[:, 0]
uy = ts_uy[:, 0]

det_uncoupled = BindingDetector(max_dim=1, baseline="max")
det_uncoupled.fit(ux, uy, subsample=1500, seed=42)

print(f"Uncoupled binding score: {det_uncoupled.binding_score():.2f}")
print(f"Coupled binding score:   {score:.2f}")
print("\nNote: even uncoupled systems have a non-zero structural baseline.")
print("This is why surrogate testing (not raw scores) is the correct test.")

## What next?

- **Heterogeneous timescales**: See `tutorial_heterogeneous_timescales.ipynb` for coupling detection between systems with different intrinsic frequencies
- **Coupling sweep**: See `fig1_coupling_sweep.ipynb` for how binding score varies with coupling strength
- **EEG application**: See `fig7_real_eeg.ipynb` for transition detection on real neural data
- **Benchmarking**: See `fig3_benchmark_overlay.ipynb` for comparison with transfer entropy, PAC, and CRQA
- **API docs**: https://musicofhel.github.io/att-docs/